# Feature Engineering - Lending Club Dataset

## 1. Cargar Librerías y Datos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
import re

# Configuración para visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
dataset_path = '/home/jules/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3/accepted_2007_to_2018Q4.csv.gz'
df = pd.DataFrame() # Inicializar df vacío
try:
    # Limit rows for memory efficiency during automated runs
    df = pd.read_csv(dataset_path, compression='gzip', low_memory=False, nrows=200000) 
    print(f"Dataset cargado exitosamente (primeras 200,000 filas). Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: El archivo no se encontró en la ruta: {dataset_path}")
except Exception as e:
    print(f"Ocurrió un error al cargar el dataset: {e}")

## 2. Definir Variable Objetivo (`is_default`)

In [ ]:
if not df.empty:
    # Definir los estados que consideramos como 'default'
    # Ampliamos la definición para incluir otros estados que claramente no son 'Fully Paid' o 'Current'
    default_statuses = [
        'Charged Off', 
        'Default', 
        'Does not meet the credit policy. Status:Charged Off', 
        'Late (31-120 days)' # Consideramos 'Late (31-120 days)' como default para ser más conservadores
    ]
    
    df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)
    
    print("Distribución de la variable objetivo 'is_default':")
    print(df['is_default'].value_counts(normalize=True) * 100)
    
    X = df.drop(['loan_status', 'is_default'], axis=1)
    y = df['is_default']
    print(f"\nDimensiones de X: {X.shape}, Dimensiones de y: {y.shape}")

## 3. Limpieza Inicial de Características

In [ ]:
if not X.empty:
    # 3.1. Eliminar columnas de IDs y texto libre no estructurado
    cols_to_drop_ids_text = ['id', 'member_id', 'url', 'desc', 'title', 'emp_title']
    # Verificar si las columnas existen antes de eliminarlas
    existing_cols_to_drop = [col for col in cols_to_drop_ids_text if col in X.columns]
    X = X.drop(columns=existing_cols_to_drop, errors='ignore')
    print(f"Columnas eliminadas (IDs, Texto): {existing_cols_to_drop}")
    print(f"Shape de X después de eliminar IDs/Texto: {X.shape}")

    # 3.2. Eliminar columnas con alto porcentaje de valores faltantes (basado en EDA típico)
    # Esta lista se basa en los hallazgos comunes del EDA para este dataset
    # En un escenario real, esta lista se generaría a partir de la salida del notebook EDA.
    cols_high_missing = [
        'mths_since_last_delinq', 'mths_since_last_record', 'next_pymnt_d', 
        'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 
        'verification_status_joint', 'revol_bal_joint', 
        # Columnas de hardship (generalmente muy vacías si no se usan)
        'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 
        'hardship_amount', 'hardship_start_date', 'hardship_end_date', 
        'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 
        'hardship_loan_status', 'orig_projected_additional_accrued_interest', 
        'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
        # Columnas de aplicaciones secundarias (sec_app_...)
        'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 
        'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 
        'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 
        'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med',
        'sec_app_mths_since_last_major_derog',
        # Otras que suelen tener muchos nulos o son problemáticas
        'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
        'settlement_amount', 'settlement_percentage', 'settlement_term',
        'il_util', 'mths_since_recent_bc_dlq', 'mths_since_rcnt_il',
        'all_util', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m',
        'total_cu_tl', 'inq_fi', 'inq_last_12m', 'open_rv_12m', 'open_rv_24m',
        'max_bal_bc', 'total_bal_il' # Estas últimas pueden variar, revisar EDA.
    ]
    # Verificar si las columnas existen antes de eliminarlas
    existing_cols_high_missing = [col for col in cols_high_missing if col in X.columns]
    X = X.drop(columns=existing_cols_high_missing, errors='ignore')
    print(f"\nColumnas eliminadas por alto % de NaN (ejemplos): {existing_cols_high_missing[:5]} ... Total: {len(existing_cols_high_missing)}")
    print(f"Shape de X después de eliminar columnas con alto % NaN: {X.shape}")

    # 3.3. Eliminar columnas con varianza casi nula (ej. un solo valor único)
    # 'policy_code' y 'pymnt_plan' suelen ser problemáticas.
    # 'application_type' también puede serlo si la mayoría son 'Individual'.
    cols_near_zero_var = ['policy_code', 'pymnt_plan', 'application_type'] # Asumiendo que 'application_type' es mayoritariamente 'Individual'
    # Verificar si las columnas existen antes de eliminarlas
    existing_cols_near_zero_var = [col for col in cols_near_zero_var if col in X.columns]
    X = X.drop(columns=existing_cols_near_zero_var, errors='ignore')
    print(f"\nColumnas eliminadas por varianza casi nula: {existing_cols_near_zero_var}")
    print(f"Shape de X después de eliminar columnas con varianza casi nula: {X.shape}")

## 4. Conversión de Tipos de Datos y Transformaciones

In [ ]:
if not X.empty:
    # 4.1. 'term': Limpiar y convertir a numérico
    if 'term' in X.columns:
        X['term'] = X['term'].str.extract('(\d+)') # Extraer solo los dígitos
        X['term'] = pd.to_numeric(X['term'], errors='coerce')
        print("Columna 'term' transformada.")

    # 4.2. 'emp_length': Convertir a numérico
    if 'emp_length' in X.columns:
        emp_length_mapping = {
            '< 1 year': 0,
            '1 year': 1,
            '2 years': 2,
            '3 years': 3,
            '4 years': 4,
            '5 years': 5,
            '6 years': 6,
            '7 years': 7,
            '8 years': 8,
            '9 years': 9,
            '10+ years': 10,
            'n/a': np.nan # Convertir 'n/a' a NaN para imputación posterior
        }
        X['emp_length'] = X['emp_length'].map(emp_length_mapping)
        print("Columna 'emp_length' transformada.")

    # 4.3. Columnas de Fecha
    date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
    for col in date_cols:
        if col in X.columns:
            X[col] = pd.to_datetime(X[col], errors='coerce')
            X[col + '_month'] = X[col].dt.month
            X[col + '_year'] = X[col].dt.year
            X = X.drop(columns=[col], errors='ignore') # Eliminar la columna de fecha original
    print(f"Columnas de fecha transformadas: {date_cols}")

    # Crear 'credit_history_length' en meses
    if 'issue_d_year' in X.columns and 'earliest_cr_line_year' in X.columns and \
       'issue_d_month' in X.columns and 'earliest_cr_line_month' in X.columns:
        X['credit_history_length'] = (X['issue_d_year'] - X['earliest_cr_line_year']) * 12 + \
                                     (X['issue_d_month'] - X['earliest_cr_line_month'])
        # Manejar casos donde 'earliest_cr_line' es posterior a 'issue_d' o meses son negativos
        X.loc[X['credit_history_length'] < 0, 'credit_history_length'] = 0 
        print("Característica 'credit_history_length' creada.")
        
    # 4.4. Columnas de Porcentaje a Flotante
    if 'int_rate' in X.columns:
        X['int_rate'] = X['int_rate'].astype(str).str.rstrip('%').astype('float') / 100.0
        print("Columna 'int_rate' transformada.")
    if 'revol_util' in X.columns:
        X['revol_util'] = X['revol_util'].astype(str).str.rstrip('%').astype('float') / 100.0
        print("Columna 'revol_util' transformada.")
    
    print(f"\nShape de X después de transformaciones: {X.shape}")

## 5. Imputación de Valores Faltantes

In [ ]:
if not X.empty:
    # Separar columnas numéricas y categóricas
    numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X.select_dtypes(include='object').columns.tolist()

    # Imputar numéricas con la mediana
    for col in numerical_cols:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].median())
    print(f"Valores faltantes imputados para {len(numerical_cols)} columnas numéricas (usando mediana).")

    # Imputar categóricas con la moda
    for col in categorical_cols:
        if X[col].isnull().any():
            X[col] = X[col].fillna(X[col].mode()[0]) # mode() puede devolver múltiples modas, tomar la primera
    print(f"Valores faltantes imputados para {len(categorical_cols)} columnas categóricas (usando moda).")
    
    # Verificar si quedan valores nulos
    print(f"\nTotal de valores nulos restantes en X: {X.isnull().sum().sum()}")

## 6. Codificación de Características Categóricas

In [ ]:
if not X.empty:
    # Actualizar lista de columnas categóricas después de transformaciones e imputación
    categorical_cols = X.select_dtypes(include='object').columns.tolist()
    
    if categorical_cols:
        print(f"Columnas categóricas a codificar: {categorical_cols}")
        X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dummy_na=False)
        print(f"\nShape de X después de one-hot encoding: {X.shape}")
    else:
        print("No hay columnas categóricas para codificar.")

## 7. Escalado de Características Numéricas

In [ ]:
if not X.empty:
    # Actualizar lista de columnas numéricas (todas las columnas deberían ser numéricas ahora)
    numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
    
    scaler = StandardScaler()
    X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
    print(f"Características numéricas escaladas usando StandardScaler.")
    display(X.head())

## 8. Selección de Características usando `mutual_info_classif`

In [ ]:
if not X.empty:
    # Para evitar data leakage, la selección de características se debe hacer en el set de entrenamiento.
    # Aquí, por simplicidad y para demostrar el proceso, lo haremos sobre una muestra o todo X,
    # pero en un pipeline de ML completo, esto iría después del train-test split.
    # Considerar que mutual_info_classif puede ser costoso en datasets muy anchos.
    # Tomaremos una muestra si el dataset es demasiado grande para este paso en el notebook.
    
    X_sample = X
    y_sample = y
    if X.shape[0] * X.shape[1] > 100_000_000: # Si el dataset es muy grande (e.g. >100M elementos)
        print("Tomando una muestra del 10% de los datos para mutual_info_classif debido al tamaño.")
        X_sample, _, y_sample, _ = train_test_split(X, y, test_size=0.9, stratify=y, random_state=42)
        
    print(f"Calculando scores de información mutua para {X_sample.shape[1]} características...")
    # Asegurarse que no haya NaNs infinitos o demasiado grandes que puedan venir de transformaciones previas
    X_sample = X_sample.replace([np.inf, -np.inf], np.nan)
    X_sample = X_sample.fillna(0) # Rellenar NaNs restantes con 0 (o media/mediana si es más apropiado) antes de MI

    mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42)
    mi_scores_series = pd.Series(mi_scores, index=X_sample.columns).sort_values(ascending=False)
    
    print("\nTop 10 características por Mutual Information Score:")
    display(mi_scores_series.head(10))
    
    # Seleccionar el top N de características. N=75 es un punto de partida razonable.
    # Se busca un balance entre tener suficientes predictores y reducir la dimensionalidad.
    N_FEATURES_TO_SELECT = 75
    selected_features = mi_scores_series.head(N_FEATURES_TO_SELECT).index.tolist()
    print(f"\nSe seleccionaron las top {N_FEATURES_TO_SELECT} características.")
    
    X_selected = X[selected_features]
    print(f"\nShape de X después de la selección de características: {X_selected.shape}")
    display(X_selected.head())

### Justificación para N_FEATURES_TO_SELECT = 75:
La elección de 75 características es heurística y busca un equilibrio entre:
1.  **Reducción de Dimensionalidad**: El dataset original (después de la codificación one-hot) puede tener cientos de características. Reducir este número ayuda a mitigar la maldición de la dimensionalidad, reduce el riesgo de overfitting y disminuye el tiempo de entrenamiento del modelo.
2.  **Retención de Información**: `mutual_info_classif` ayuda a identificar características que comparten más información con la variable objetivo. Al tomar las top 75, esperamos retener la mayor parte de la señal predictiva.
3.  **Complejidad del Modelo**: Un número menor de características generalmente conduce a modelos más simples y más interpretables.
En una implementación real, este número podría ajustarse mediante validación cruzada o analizando la curva de los scores de información mutua para encontrar un "codo" o punto de inflexión.

## 9. Guardar Datos Procesados

In [ ]:
if not X_selected.empty:
    processed_df = pd.concat([X_selected, y], axis=1)
    # Asegurarse de que el directorio de salida exista
    import os
    output_dir = '../data/processed/'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'processed_lending_club_data.csv')
    try:
        processed_df.to_csv(output_path, index=False)
        print(f"\nDatos procesados guardados en: {output_path}")
        print(f"Shape de los datos guardados: {processed_df.shape}")
    except Exception as e:
        print(f"Error al guardar los datos procesados: {e}")